# SpecAugment: Spectrogram Augmentation for Speech Recognition

This notebook implements and demonstrates **SpecAugment** (Park et al., 2019), a data augmentation
method that operates directly on log mel spectrograms. Originally developed for ASR at Google Brain,
we adapt it here for our **DementiaNet** dementia-detection pipeline.

---

## Background

Deep learning models for speech tasks tend to overfit, especially with limited clinical datasets
(Chiu et al., 2018). Our DementiaNet baseline uses `facebook/wav2vec2-base` fine-tuned on
deterministic 30-second clips extracted from DementiaBank recordings, with speaker-level splits
to prevent data leakage. With only ~325 training clips and ~80 validation clips, overfitting
is a primary concern.

SpecAugment addresses this by augmenting the *feature representation* rather than the raw
waveform, making it computationally cheap and compatible with online training. It requires
no external data — only stochastic masking of the spectrogram.

The method consists of three deformations applied to the log mel spectrogram:

| Deformation | Description | Motivation |
|---|---|---|
| **Time Warping** | Non-linear displacement along the time axis | Robustness to speaking rate variation |
| **Frequency Masking** | Zero out contiguous frequency bands | Robustness to partial frequency loss |
| **Time Masking** | Zero out contiguous time steps | Robustness to brief dropouts in speech |

Park et al. showed that these simple transforms convert ASR from an **overfitting** to an
**underfitting** problem — meaning bigger models and longer training schedules yield continued gains.

### Predefined Augmentation Policies (Park et al., 2019, Table 1)

| Policy | W | F | mF | T | p | mT |
|--------|---|---|---|---|---|---|
| None   | 0 | 0 | — | 0 | — | — |
| LB (LibriSpeech Basic) | 80 | 27 | 1 | 100 | 1.0 | 1 |
| LD (LibriSpeech Double) | 80 | 27 | 2 | 100 | 1.0 | 2 |
| SM (Switchboard Mild) | 40 | 15 | 2 | 70 | 0.2 | 2 |
| SS (Switchboard Strong) | 40 | 27 | 2 | 70 | 0.2 | 2 |

We use the **SM** policy as our baseline, with the adaptive time-mask cap `p=0.2` that prevents
masking more than 20% of time frames — appropriate for our shorter clinical recordings
(typically 20–90 seconds, segmented into 30-second clips).

### How SpecAugment Fits Our Pipeline

```
Raw Audio (audio/dementia/, audio/nodementia/)
    → Dataset construction + speaker splits (00_build_scientific_dataset)
    → Deterministic 30s clip expansion (01_finalize_validated_dataset)
    → SpecAugment on training clips only (this notebook)
    → wav2vec2-base fine-tuning (02_train_scientific_baseline)
    → Evaluation (03_evaluate_scientific_baseline)
```

Augmentation is applied **only to training clips** — validation and test sets are never
augmented, preserving an unbiased evaluation protocol.

### References

- Park, D. S., Chan, W., Zhang, Y., Chiu, C.-C., Zoph, B., Cubuk, E. D., & Le, Q. V. (2019).
  *SpecAugment: A Simple Data Augmentation Method for Automatic Speech Recognition.*
  Interspeech 2019. arXiv:1904.08779
- DeVries, T. & Taylor, G. (2017). *Improved Regularization of Convolutional Neural Networks
  with Cutout.* arXiv:1708.04552 — inspiration for masking-based augmentation.
- Ko, T., Peddinti, V., Povey, D., & Khudanpur, S. (2015). *Audio Augmentation for Speech
  Recognition.* Interspeech 2015 — speed perturbation on raw audio.
- Cubuk, E. D., Zoph, B., Mané, D., Vasudevan, V., & Le, Q. V. (2019). *AutoAugment: Learning
  Augmentation Policies from Data.* CVPR 2019 — learned augmentation in the image domain.
- Baevski, A., Zhou, H., Mohamed, A., & Auli, M. (2020). *wav2vec 2.0: A Framework for
  Self-Supervised Learning of Speech Representations.* NeurIPS 2020 — the pretrained
  encoder used in our DementiaNet baseline.

In [ ]:
import os
import json
import numpy as np
import torch
import torchaudio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import IPython.display as ipd

from pathlib import Path

print(f'PyTorch    : {torch.__version__}')
print(f'torchaudio : {torchaudio.__version__}')

In [ ]:
# ── Colab users: mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths (aligned with 01_finalize_validated_dataset.ipynb, March 2026) ─────
import pandas as pd

BASE_DIR   = Path('/content/drive/MyDrive/CS7357/Project/data')
OUTPUT_DIR = BASE_DIR / 'clean_dataset_March_2026_alt_data'

TRAIN_MANIFEST = OUTPUT_DIR / 'manifests' / 'train_dm.csv'
VALID_MANIFEST = OUTPUT_DIR / 'manifests' / 'valid_dm.csv'
TEST_MANIFEST  = OUTPUT_DIR / 'manifests' / 'test_dm.csv'

# Load manifests produced by notebook 01_finalize
train_df = pd.read_csv(TRAIN_MANIFEST, sep='\t')
valid_df = pd.read_csv(VALID_MANIFEST, sep='\t')
test_df  = pd.read_csv(TEST_MANIFEST,  sep='\t')

print(f'Train clips : {len(train_df)}')
print(f'Valid clips  : {len(valid_df)}')
print(f'Test clips   : {len(test_df)}')
print(f'\nManifest columns: {list(train_df.columns)}')
print(f'\nTrain label distribution:')
print(train_df['label'].value_counts().to_string())
train_df.head()

### Loading Deterministic 30-Second Clips

Notebook `01_finalize_validated_dataset` segments each recording into fixed 30-second clips:
- **Training**: stride = 20s (33% overlap, more clips per recording)
- **Validation / Test**: stride = 30s (non-overlapping, clean evaluation)

Each manifest row specifies `start_sec` and `clip_sec` so we load the exact segment.
Recordings shorter than 30s produce a single clip, zero-padded at load time.

In [ ]:
CLIP_SECONDS = 30
TARGET_SR    = 16_000  # DementiaBank recordings are 16 kHz

def load_clip(row):
    """
    Load a deterministic 30-second clip from a manifest row.

    Uses start_sec and clip_sec to extract the exact segment.
    Recordings shorter than clip_sec are zero-padded to the target length.
    Returns a mono waveform tensor and the sample rate.
    """
    path   = row['path']
    start  = row['start_sec']
    clip_s = row['clip_sec']

    frame_offset = int(start * TARGET_SR)
    num_frames   = int(clip_s * TARGET_SR)

    waveform, sr = torchaudio.load(path, frame_offset=frame_offset, num_frames=num_frames)

    # Collapse to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Zero-pad if the recording was shorter than the clip window
    if waveform.shape[1] < num_frames:
        pad = num_frames - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, pad))

    return waveform, sr


# Load a sample training clip for demonstrations
sample_row = train_df.iloc[0]
waveform, sr = load_clip(sample_row)
sample_label = sample_row['label']
sample_file  = sample_row['file']

print(f'Sample file : {sample_file}')
print(f'Label       : {sample_label}')
print(f'Speaker     : {sample_row["speaker"]}')
print(f'Start       : {sample_row["start_sec"]}s')
print(f'Clip length : {sample_row["clip_sec"]}s')
print(f'Sample rate : {sr} Hz')
print(f'Waveform    : {waveform.shape}  ({waveform.shape[1] / sr:.1f}s)')

In [ ]:
N_FFT      = 1024
HOP_LENGTH = 256
N_MELS     = 80

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
)


def compute_log_mel(wav: torch.Tensor) -> torch.Tensor:
    """Compute log mel spectrogram: (1, T) waveform → (n_mels, tau) log-power."""
    mel = mel_transform(wav).squeeze(0)          # (n_mels, tau)
    return torch.log(mel + 1e-9)


log_mel = compute_log_mel(waveform)
print(f'Log mel spectrogram shape: {log_mel.shape}  (freq_bins={log_mel.shape[0]}, time_frames={log_mel.shape[1]})')

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
ax.set_xlabel('Time Frame')
ax.set_ylabel('Mel Bin')
ax.set_title(f'Log Mel Spectrogram — {SAMPLE_PATH.stem}')
plt.colorbar(ax.images[0], ax=ax, label='Log Power')
plt.tight_layout()
plt.show()

---
## 2. Frequency Masking

From Park et al. (2019), §2:

> Frequency masking is applied so that $f$ consecutive mel frequency channels
> $[f_0, f_0 + f)$ are masked, where $f$ is first chosen from a uniform distribution
> from 0 to the frequency mask parameter $F$, and $f_0$ is chosen from $[0, \nu - f)$.

This forces the model to make predictions without relying on any single frequency band —
analogous to **Cutout** (DeVries & Taylor, 2017) in the image domain.

Since the log mel spectrogram is mean-normalized, setting masked regions to zero is equivalent
to setting them to the mean value.

In [ ]:
def frequency_mask(spec: torch.Tensor, F: int, num_masks: int = 1) -> torch.Tensor:
    """
    Apply frequency masking to a spectrogram.

    Args:
        spec      : (freq_bins, time_frames) spectrogram
        F         : maximum width of each frequency mask (in bins)
        num_masks : number of independent frequency masks to apply (mF)

    Returns:
        Masked copy of the spectrogram.
    """
    masked = spec.clone()
    nu = spec.shape[0]  # number of frequency channels

    for _ in range(num_masks):
        f  = torch.randint(0, F + 1, (1,)).item()         # mask width
        f0 = torch.randint(0, max(1, nu - f), (1,)).item()  # mask start
        masked[f0 : f0 + f, :] = 0.0

    return masked


# Demonstrate with F=15 (SM policy) and F=27 (LD/SS policy)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original')

for i, (F_val, label) in enumerate([(15, 'F=15, mF=2 (SM)'), (27, 'F=27, mF=2 (LD/SS)')], 1):
    masked = frequency_mask(log_mel, F=F_val, num_masks=2)
    axes[i].imshow(masked.numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[i].set_title(f'Frequency Masked — {label}')

for ax in axes:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle('Frequency Masking', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Time Masking

From Park et al. (2019), §2:

> Time masking is applied so that $t$ consecutive time steps $[t_0, t_0 + t)$ are masked,
> where $t$ is first chosen from a uniform distribution from 0 to the time mask parameter $T$,
> and $t_0$ is chosen from $[0, \tau - t)$.
>
> We introduce an upper bound on the time mask so that a time mask cannot be wider than
> $p$ times the number of time steps.

The adaptive cap $p$ is important for shorter audio clips (like our clinical recordings).
With $p = 0.2$, no single mask can exceed 20% of the total duration, preventing excessive
information loss.

In [ ]:
def time_mask(spec: torch.Tensor, T: int, p: float = 1.0, num_masks: int = 1) -> torch.Tensor:
    """
    Apply time masking to a spectrogram.

    Args:
        spec      : (freq_bins, time_frames) spectrogram
        T         : maximum width of each time mask (in frames)
        p         : upper-bound ratio — mask width cannot exceed p * tau
        num_masks : number of independent time masks to apply (mT)

    Returns:
        Masked copy of the spectrogram.
    """
    masked = spec.clone()
    tau = spec.shape[1]  # number of time frames

    adaptive_T = min(T, int(p * tau))  # cap per Park et al.
    if adaptive_T <= 0:
        return masked

    for _ in range(num_masks):
        t  = torch.randint(0, adaptive_T + 1, (1,)).item()    # mask width
        t0 = torch.randint(0, max(1, tau - t), (1,)).item()     # mask start
        masked[:, t0 : t0 + t] = 0.0

    return masked


# Demonstrate p=1.0 (uncapped) vs p=0.2 (SM policy)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original')

for i, (p_val, label) in enumerate([(1.0, 'T=70, p=1.0 (uncapped)'), (0.2, 'T=70, p=0.2 (SM)')], 1):
    masked = time_mask(log_mel, T=70, p=p_val, num_masks=2)
    axes[i].imshow(masked.numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[i].set_title(f'Time Masked — {label}')

for ax in axes:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle('Time Masking — Effect of Adaptive Cap p', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Time Warping

From Park et al. (2019), §2:

> A random point along the horizontal line passing through the center of the image
> within the time steps $(W, \tau - W)$ is to be warped either to the left or right
> by a distance $w$ chosen from a uniform distribution from 0 to the time warp
> parameter $W$ along that line.

Time warping simulates natural variation in speaking rate. It is the most computationally
expensive of the three transforms and Park et al. noted it has the **smallest effect** on
final WER (Table 6 of the paper). For our pipeline, we implement it for completeness but
note that it can be safely omitted under compute constraints.

We implement a simplified version using `torch.nn.functional.interpolate` to locally
stretch/compress the spectrogram around a random warp point.

In [ ]:
import torch.nn.functional as F_torch


def time_warp(spec: torch.Tensor, W: int = 40) -> torch.Tensor:
    """
    Apply simplified time warping to a spectrogram.

    Picks a random column in (W, tau-W), then displaces it by a random
    offset w in [-W, W]. The left and right halves are independently
    resampled to fill the original width.

    Args:
        spec : (freq_bins, time_frames) spectrogram
        W    : time warp parameter — max displacement in frames

    Returns:
        Warped copy of the spectrogram (same shape).
    """
    tau = spec.shape[1]
    if tau <= 2 * W + 1:
        return spec.clone()  # clip too short to warp

    # Random source point and displacement
    src_pt = torch.randint(W, tau - W, (1,)).item()
    w      = torch.randint(-W, W + 1, (1,)).item()
    dst_pt = src_pt + w
    dst_pt = max(1, min(dst_pt, tau - 1))  # keep within bounds

    # Split, resample each half, concatenate
    left  = spec[:, :src_pt].unsqueeze(0)   # (1, freq, src_pt)
    right = spec[:, src_pt:].unsqueeze(0)    # (1, freq, tau - src_pt)

    left_warped  = F_torch.interpolate(left,  size=dst_pt,       mode='linear', align_corners=False)
    right_warped = F_torch.interpolate(right, size=tau - dst_pt, mode='linear', align_corners=False)

    return torch.cat([left_warped.squeeze(0), right_warped.squeeze(0)], dim=1)


# Demonstrate
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original')

for i in range(1, 3):
    warped = time_warp(log_mel, W=40)
    axes[i].imshow(warped.numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[i].set_title(f'Time Warped (W=40) — Example {i}')

for ax in axes:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle('Time Warping', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Full SpecAugment Pipeline

Combining all three transforms into a single function. The order follows the paper:
**time warp → frequency mask → time mask**.

We provide two entry points:
1. `specaugment_mel()` — operates on a log mel spectrogram (for visualization/analysis)
2. `specaugment_wav()` — operates on a raw waveform via STFT domain (for producing augmented `.wav` files)

In [ ]:
# ── Policy Configurations ────────────────────────────────────────────────────
POLICIES = {
    'None': dict(W=0,  F=0,  mF=0, T=0,   p=0.0, mT=0),
    'LB':   dict(W=80, F=27, mF=1, T=100, p=1.0, mT=1),
    'LD':   dict(W=80, F=27, mF=2, T=100, p=1.0, mT=2),
    'SM':   dict(W=40, F=15, mF=2, T=70,  p=0.2, mT=2),
    'SS':   dict(W=40, F=27, mF=2, T=70,  p=0.2, mT=2),
}


def specaugment_mel(spec: torch.Tensor, policy: str = 'SM') -> torch.Tensor:
    """
    Apply the full SpecAugment pipeline to a log mel spectrogram.

    Args:
        spec   : (freq_bins, time_frames) log mel spectrogram
        policy : one of 'None', 'LB', 'LD', 'SM', 'SS'

    Returns:
        Augmented spectrogram (same shape).
    """
    cfg = POLICIES[policy]
    out = spec.clone()

    # 1. Time warping
    if cfg['W'] > 0:
        out = time_warp(out, W=cfg['W'])

    # 2. Frequency masking
    if cfg['mF'] > 0 and cfg['F'] > 0:
        out = frequency_mask(out, F=cfg['F'], num_masks=cfg['mF'])

    # 3. Time masking
    if cfg['mT'] > 0 and cfg['T'] > 0:
        out = time_mask(out, T=cfg['T'], p=cfg['p'], num_masks=cfg['mT'])

    return out


def specaugment_wav(waveform: torch.Tensor, sample_rate: int,
                    n_fft: int = 1024, hop_length: int = 256,
                    policy: str = 'SM') -> torch.Tensor:
    """
    Apply SpecAugment in the STFT domain and reconstruct the waveform.

    This version works on the magnitude spectrogram (not mel-scaled) so
    that phase-based reconstruction via iSTFT is straightforward.

    Pipeline: waveform → STFT → apply masks on magnitude → iSTFT → waveform

    Args:
        waveform    : (C, T) tensor from torchaudio.load()
        sample_rate : sample rate (passed through)
        n_fft       : FFT window size
        hop_length  : STFT hop
        policy      : augmentation policy name

    Returns:
        (1, T) mono augmented waveform tensor
    """
    cfg = POLICIES[policy]

    # Collapse to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    window = torch.hann_window(n_fft)
    stft   = torch.stft(waveform[0], n_fft=n_fft, hop_length=hop_length,
                         window=window, return_complex=True)
    mag   = stft.abs()
    phase = stft.angle()
    n_freq, tau = mag.shape

    # Frequency masking
    for _ in range(cfg['mF']):
        f  = torch.randint(1, cfg['F'] + 1, (1,)).item()
        f0 = torch.randint(0, max(1, n_freq - f), (1,)).item()
        mag[f0 : f0 + f, :] = 0.0

    # Time masking
    adaptive_T = min(cfg['T'], int(cfg['p'] * tau))
    if adaptive_T > 0:
        for _ in range(cfg['mT']):
            t  = torch.randint(1, adaptive_T + 1, (1,)).item()
            t0 = torch.randint(0, max(1, tau - t), (1,)).item()
            mag[:, t0 : t0 + t] = 0.0

    # Reconstruct
    masked_stft = mag * torch.exp(1j * phase)
    augmented   = torch.istft(masked_stft, n_fft=n_fft, hop_length=hop_length,
                              window=window, length=waveform.shape[1])
    return augmented.unsqueeze(0)


print('Pipeline functions defined: specaugment_mel(), specaugment_wav()')

---
## 6. Policy Comparison — Visual

Reproducing the style of Figure 2 from Park et al. (2019): the same input augmented
under each of the predefined policies.

In [ ]:
policy_names = ['None', 'LB', 'LD', 'SM', 'SS']
n_policies = len(policy_names)

fig, axes = plt.subplots(n_policies, 1, figsize=(14, 3 * n_policies))

for ax, name in zip(axes, policy_names):
    aug = specaugment_mel(log_mel, policy=name)
    ax.imshow(aug.numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'Policy: {name}', fontsize=12)
    ax.set_ylabel('Mel Bin')

axes[-1].set_xlabel('Time Frame')
plt.suptitle('SpecAugment Policies Applied to the Same Input', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Ablation Study: Individual Augmentation Effects

Park et al. (2019), Table 6, showed that removing each transform individually degrades
performance, but time warping has the smallest contribution:

| Config | test-other WER |
|--------|---------------|
| Full (LB) | 10.0% |
| No time warp | 10.1% |
| No freq mask | 11.0% |
| No time mask | 10.9% |

Below we visualize each augmentation applied in isolation to build intuition.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Original
axes[0, 0].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0, 0].set_title('Original (No Augmentation)')

# Time warp only
tw = time_warp(log_mel, W=40)
axes[0, 1].imshow(tw.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0, 1].set_title('Time Warp Only (W=40)')

# Frequency mask only
fm = frequency_mask(log_mel, F=15, num_masks=2)
axes[1, 0].imshow(fm.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1, 0].set_title('Frequency Mask Only (F=15, mF=2)')

# Time mask only
tm = time_mask(log_mel, T=70, p=0.2, num_masks=2)
axes[1, 1].imshow(tm.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1, 1].set_title('Time Mask Only (T=70, p=0.2, mT=2)')

for ax in axes.flat:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle('Ablation: Individual SpecAugment Transforms (cf. Park et al. Table 6)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Audio Comparison: Original vs Augmented

Since our pipeline (notebook 01) applies SpecAugment in the STFT domain and reconstructs
the waveform, we can listen to the effect directly. The augmented audio should sound similar
to the original but with brief silences (time masks) or muffled frequency bands (freq masks).

In [ ]:
print('=== Original ===')
ipd.display(ipd.Audio(waveform.numpy(), rate=sr))

for policy_name in ['SM', 'SS', 'LD']:
    aug_wav = specaugment_wav(waveform, sr, policy=policy_name)
    print(f'\n=== SpecAugment — {policy_name} ===')
    ipd.display(ipd.Audio(aug_wav.numpy(), rate=sr))

---
## 9. Stochastic Variation — Multiple Augmented Copies

Each call to SpecAugment produces a unique augmentation because mask positions and widths
are sampled randomly. Below we generate 6 augmented versions of the same clip to demonstrate
the diversity — this is the same strategy used in notebook 01 where we create multiple copies
per training sample.

In [ ]:
n_copies = 6
fig, axes = plt.subplots(2, 3, figsize=(18, 7))

for i, ax in enumerate(axes.flat):
    aug = specaugment_mel(log_mel, policy='SM')
    ax.imshow(aug.numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'Copy {i + 1}', fontsize=11)
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle(f'{n_copies} Independent SM-Policy Augmentations of the Same Input', fontsize=13)
plt.tight_layout()
plt.show()

---
## 10. DementiaNet Configuration Rationale

Our scientific baseline (see `02_train_scientific_baseline.ipynb`) fine-tunes
`facebook/wav2vec2-base` with a frozen feature encoder on deterministic 30-second clips
segmented from DementiaBank recordings. The baseline achieves reasonable performance but is
prone to overfitting given the small dataset (~325 train clips, ~80 validation clips).

We chose the **SM (Switchboard Mild)** policy with class-dependent augmentation multipliers:

| Parameter | Value | Reasoning |
|-----------|-------|-----------|
| Policy | SM | Conservative — our 30s clips yield ~1875 STFT frames at hop=256, shorter than LibriSpeech |
| F | 15 | Moderate freq masking — preserves prosodic and spectral cues important for dementia detection |
| mF | 2 | Two independent masks per sample |
| T | 70 | Generous time mask parameter |
| p | 0.2 | Adaptive cap — at 1875 frames, max mask width = 375 frames (~6s). Prevents excessive masking |
| mT | 2 | Two independent masks per sample |
| Copies (dementia) | 6 | Oversample minority class to address class imbalance (101 dem vs 105 nodem originals) |
| Copies (nodementia) | 2 | Light augmentation for majority class |

### Why SM and Not LD?

The LibriSpeech policies (LB/LD) use `p=1.0` (uncapped time masks) and `T=100`, meaning a
single mask can wipe out 100 consecutive frames (~1.6s) regardless of clip length. For our
30-second clips this would still be <10% of total duration, but the SM policy's `p=0.2` cap
provides an additional safety margin that prevents rare but aggressive mask combinations from
destroying too much clinical signal.

The SM policy's `F=15` (vs LD's `F=27`) is also more conservative, which matters because
dementia detection relies on subtle spectral features (voice quality, formant transitions)
that broader frequency masks might obscure.

### Augmentation as Class Balancing

The asymmetric copy count (6x dementia vs 2x nodementia) serves dual purposes:
1. **Data augmentation** — regularizes training and combats overfitting on a small clinical dataset
2. **Class balancing** — produces a near-balanced training set without discarding majority-class samples

This complements the class-weighted cross entropy loss already used in the baseline trainer
(`WeightedTrainer`), providing both input-level and loss-level strategies for handling imbalance.

### Connection to Park et al.'s Key Finding

Park et al. observed that SpecAugment converts overfitting into underfitting. This is exactly
what we need: our baseline overfits the small DementiaBank training set. By making the effective
training distribution harder (via masking), the model is forced to learn more generalizable
representations from the wav2vec2 encoder — rather than memorizing speaker-specific artifacts.

In [ ]:
# DementiaNet SpecAugment config
DEMENTIANET_CONFIG = {
    'policy': 'SM',
    'n_fft': 1024,
    'hop_length': 256,
    'F': 15,
    'mF': 2,
    'T': 70,
    'p': 0.2,
    'mT': 2,
    'num_augments': {'dementia': 6, 'nodementia': 2},
}

print('DementiaNet SpecAugment Configuration')
print('=' * 45)
for k, v in DEMENTIANET_CONFIG.items():
    print(f'  {k:20s}: {v}')

---
## 11. Augmentation Loop — Produce Augmented Training Clips

This section applies SpecAugment to every training clip and saves the augmented
waveforms as `.wav` files. The output is:

1. **Augmented `.wav` files** in `augmented_clips/` on Google Drive
2. **`train_dm_augmented.csv`** — a new manifest combining originals + augmented clips

Augmentation is applied **only to training clips**. Validation and test manifests
are copied unchanged.

Each training clip gets `num_augments[label]` augmented copies:
- dementia clips: 6 copies (oversample minority class)
- nodementia clips: 2 copies (light augmentation for majority class)

In [ ]:
import shutil
from tqdm.notebook import tqdm

# Output directories
AUG_AUDIO_DIR   = OUTPUT_DIR / 'augmented_clips'
AUG_MANIFEST    = OUTPUT_DIR / 'manifests' / 'train_dm_augmented.csv'

# Clean previous augmented output
if AUG_AUDIO_DIR.exists():
    shutil.rmtree(AUG_AUDIO_DIR)
AUG_AUDIO_DIR.mkdir(parents=True)

print(f'Augmented audio dir : {AUG_AUDIO_DIR}')
print(f'Augmented manifest  : {AUG_MANIFEST}')
print(f'Training clips      : {len(train_df)}')
print()

for label, n in DEMENTIANET_CONFIG['num_augments'].items():
    count = len(train_df[train_df['label'] == label])
    print(f'  {label:12s}: {count} clips x {n} augmented copies = {count * n} new clips')

total_aug = sum(
    len(train_df[train_df['label'] == lbl]) * n
    for lbl, n in DEMENTIANET_CONFIG['num_augments'].items()
)
print(f'\nTotal augmented clips to generate: {total_aug}')
print(f'Final training set size (orig + aug): {len(train_df) + total_aug}')

In [ ]:
augmented_rows = []
skipped = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Augmenting training clips'):
    try:
        clip_wav, clip_sr = load_clip(row)
        n_aug = DEMENTIANET_CONFIG['num_augments'][row['label']]

        for aug_i in range(n_aug):
            aug_wav = specaugment_wav(clip_wav, clip_sr, policy=DEMENTIANET_CONFIG['policy'])

            # Build a unique filename: originalname_clipN_augM.wav
            stem = Path(row['file']).stem
            aug_name = f"{stem}_clip{int(row['clip_idx'])}_aug{aug_i}.wav"
            aug_path = AUG_AUDIO_DIR / aug_name

            torchaudio.save(str(aug_path), aug_wav, clip_sr)

            augmented_rows.append({
                'file':         aug_name,
                'label':        row['label'],
                'path':         str(aug_path),
                'speaker':      row['speaker'],
                'duration_sec': row['duration_sec'],
                'start_sec':    0.0,        # augmented clip is self-contained
                'clip_sec':     float(CLIP_SECONDS),
                'clip_idx':     aug_i,
            })

    except Exception as e:
        skipped.append((row['file'], str(e)))

print(f'\nAugmented clips created: {len(augmented_rows)}')
if skipped:
    print(f'Skipped due to errors : {len(skipped)}')
    for f, e in skipped[:5]:
        print(f'  {f}: {e}')

---
## 12. Save Augmented Manifest

Combine originals + augmented clips into `train_dm_augmented.csv`.
This manifest can be used directly by the training notebook (02) in place of `train_dm.csv`.

In [ ]:
aug_df = pd.DataFrame(augmented_rows)

# Combine originals + augmented
train_augmented_df = pd.concat([train_df, aug_df], ignore_index=True)

print('=== Augmented Training Set ===')
print(f'Original clips  : {len(train_df)}')
print(f'Augmented clips : {len(aug_df)}')
print(f'Total           : {len(train_augmented_df)}')
print()
print('Label distribution (originals + augmented):')
print(train_augmented_df['label'].value_counts().to_string())
print()

# Save — tab-delimited to match existing manifest format
manifest_cols = ['file', 'label', 'path', 'speaker', 'duration_sec', 'start_sec', 'clip_sec', 'clip_idx']
train_augmented_df[manifest_cols].to_csv(AUG_MANIFEST, sep='\t', index=False)

print(f'Saved: {AUG_MANIFEST}')
print(f'       {len(train_augmented_df)} rows')
print()
print('Validation and test manifests are unchanged:')
print(f'  {VALID_MANIFEST}  ({len(valid_df)} clips)')
print(f'  {TEST_MANIFEST}   ({len(test_df)} clips)')

---
## 13. Verify Augmented Output

In [ ]:
# Reload and verify the saved manifest
verify_df = pd.read_csv(AUG_MANIFEST, sep='\t')
print(f'Reloaded manifest: {len(verify_df)} rows')
print(f'Columns: {list(verify_df.columns)}')
print()
print('Label distribution:')
print(verify_df['label'].value_counts().to_string())
print()

# Verify no speaker leakage into val/test
aug_speakers = set(verify_df['speaker'])
val_speakers = set(valid_df['speaker'])
test_speakers = set(test_df['speaker'])
print(f'Train speakers in valid: {aug_speakers & val_speakers}')
print(f'Train speakers in test : {aug_speakers & test_speakers}')
print()

# Spot-check: load one augmented clip and visualize
aug_sample = aug_df.iloc[0]
aug_wav_check, aug_sr = torchaudio.load(aug_sample['path'])
print(f'Spot-check: {aug_sample["file"]}')
print(f'  Shape: {aug_wav_check.shape}  Duration: {aug_wav_check.shape[1]/aug_sr:.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Original (first training clip)
orig_mel = compute_log_mel(waveform)
axes[0].imshow(orig_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title(f'Original — {sample_file}')

# Augmented
aug_mel = compute_log_mel(aug_wav_check)
axes[1].imshow(aug_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title(f'Augmented — {aug_sample["file"]}')

for ax in axes:
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Mel Bin')

plt.suptitle('Verification: Original vs Saved Augmented Clip', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated the three components of SpecAugment:

1. **Time Warping** — local stretching/compression of the time axis (smallest effect per ablation)
2. **Frequency Masking** — zeroing contiguous mel bands (forces robustness to missing frequencies)
3. **Time Masking** — zeroing contiguous time frames (forces robustness to brief speech gaps)

Key takeaways from Park et al. (2019):
- SpecAugment is applied to the spectrogram, not raw audio — making it computationally cheap
- It converts overfitting into underfitting, enabling gains from bigger models and longer training
- The adaptive time-mask cap `p` is critical for shorter audio (like our 30-second clinical clips)
- Time warping can be omitted first under compute constraints with minimal accuracy loss

### Integration with DementiaNet

| Pipeline Step | Notebook | Role |
|---|---|---|
| Dataset cleaning & silence trimming | `00-clean-dataset.ipynb` | Prepares raw audio |
| Dataset construction & speaker splits | `00_build_scientific_dataset.ipynb` | Deterministic 30s clips, speaker-level splits |
| SpecAugment theory & implementation | **This notebook** (`04-specaugment.ipynb`) | Reference & visualization |
| Augmented data preparation | `01-prepare-augmented-data_current.ipynb` | Applies `specaugment_wav()` to training clips |
| Baseline training (no augmentation) | `02_train_scientific_baseline.ipynb` | wav2vec2-base with frozen encoder, class-weighted loss |
| Augmented training | `02-finetune-on-augmented-dataset_current.ipynb` | Same architecture, trained on augmented data |
| Evaluation | `03-eval-augmented-dataset_current.ipynb` | Test set evaluation |

The augmentation functions defined here (`specaugment_wav`, `specaugment_mel`) are the same
implementations used in **notebook 01** to produce the augmented training set for DementiaNet
fine-tuning.